<a href="https://colab.research.google.com/github/liptonmn/ceramics-demand-forecasting/blob/fix-duplicate-registration-v2/MVPWeek5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import the Colab upload tool and the libraries needed in the notebook
from google.colab import files
import pandas as pd
import os
import shutil

# Open a file-selection window to upload train.csv and test.csv
uploaded = files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv


In [2]:
# Display the files currently available in the Colab environment
print(os.listdir("/content"))

['.config', 'test.csv', 'train.csv', 'sample_data']


In [3]:
# Create a folder called raw_data to hold unchanged copies of the CSV files
os.makedirs("raw_data", exist_ok=True)

# Copy the uploaded files into the raw_data folder
shutil.copy("train.csv", "raw_data/train.csv")
shutil.copy("test.csv", "raw_data/test.csv")

# Confirm that the raw copies were created
print("Files saved in the raw_data folder:")
print(os.listdir("raw_data"))

Files saved in the raw_data folder:
['test.csv', 'train.csv']


In [4]:
# Load the raw copies of the CSV files
train_df = pd.read_csv("raw_data/train.csv")
test_df = pd.read_csv("raw_data/test.csv")

# Print the number of rows and columns in each file
print("Train data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

# Show the first five rows of each dataset
display(train_df.head())
display(test_df.head())

Train data shape: (913000, 4)
Test data shape: (45000, 4)


,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


,id,date,store,item
0,0,2018-01-01,1,1
1,1,2018-01-02,1,1
2,2,2018-01-03,1,1
3,3,2018-01-04,1,1
4,4,2018-01-05,1,1


In [5]:
# Print the column names in both datasets
print("Train columns:")
print(train_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

# Check whether the sales column is in the training data
if "sales" in train_df.columns:
    print("\nPASS: The sales column is in train.csv.")
else:
    print("\nWARNING: The sales column is missing from train.csv.")

# Check whether the sales column is not in the test data
if "sales" not in test_df.columns:
    print("PASS: The sales column is not in test.csv.")
else:
    print("WARNING: The sales column is also in test.csv.")

# Compare the features used for modeling
# The test-file id column is expected because it identifies each forecast row
train_features = set(train_df.columns) - {"sales"}
test_features = set(test_df.columns) - {"id"}

if train_features == test_features:
    print("PASS: Train and test have matching modeling feature columns.")
    print("NOTE: The test-only 'id' column is expected and should be kept for final forecast output.")
else:
    print("WARNING: Train and test modeling feature columns do not match.")

    print("Only in train:")
    print(train_features - test_features)

    print("Only in test:")
    print(test_features - train_features)

Train columns:
['date', 'store', 'item', 'sales']

Test columns:
['id', 'date', 'store', 'item']

PASS: The sales column is in train.csv.
PASS: The sales column is not in test.csv.
PASS: Train and test have matching modeling feature columns.
NOTE: The test-only 'id' column is expected and should be kept for final forecast output.


In [6]:
# Count missing values in each training-data column
print("Missing values in train.csv:")
display(train_df.isnull().sum())

# Count missing values in each test-data column
print("Missing values in test.csv:")
display(test_df.isnull().sum())

# Print the total number of missing values in each dataset
print("\nTotal missing values in train.csv:", train_df.isnull().sum().sum())
print("Total missing values in test.csv:", test_df.isnull().sum().sum())

Missing values in train.csv:


,0
date,0
store,0
item,0
sales,0


Missing values in test.csv:


,0
id,0
date,0
store,0
item,0



Total missing values in train.csv: 0
Total missing values in test.csv: 0


In [7]:
# Count duplicate rows in each dataset
train_duplicates = train_df.duplicated().sum()
test_duplicates = test_df.duplicated().sum()

print("Duplicate rows in train.csv:", train_duplicates)
print("Duplicate rows in test.csv:", test_duplicates)

# If duplicates exist, show a few examples from the training data
if train_duplicates > 0:
    print("\nExamples of duplicate rows in train.csv:")
    display(train_df[train_df.duplicated(keep=False)].head())

Duplicate rows in train.csv: 0
Duplicate rows in test.csv: 0


In [8]:
# Convert sales to numeric values only for checking
# Invalid text values will become NaN in sales_numeric
sales_numeric = pd.to_numeric(train_df["sales"], errors="coerce")

# Count missing sales values in the original data
missing_sales = train_df["sales"].isnull().sum()

# Count values that could not be converted to numbers
non_numeric_sales = (
    train_df["sales"].notnull() & sales_numeric.isnull()
).sum()

# Count negative sales values
negative_sales = (sales_numeric < 0).sum()

# Print the results of the sales checks
print("Missing sales values:", missing_sales)
print("Non-numeric sales values:", non_numeric_sales)
print("Negative sales values:", negative_sales)

# Show rows with missing, non-numeric, or negative sales values
invalid_sales = (
    train_df["sales"].isnull()
    | sales_numeric.isnull()
    | (sales_numeric < 0)
)

if invalid_sales.sum() > 0:
    print("\nRows with invalid sales values:")
    display(train_df[invalid_sales].head())
else:
    print("\nPASS: No missing, non-numeric, or negative sales values were found.")

Missing sales values: 0
Non-numeric sales values: 0
Negative sales values: 0

PASS: No missing, non-numeric, or negative sales values were found.


In [9]:
# Look for column names that include the word "date" or "day"
possible_date_columns = []

for column in train_df.columns:
    if "date" in column.lower() or "day" in column.lower():
        possible_date_columns.append(column)

print("Possible date columns:")
print(possible_date_columns)

Possible date columns:
['date']


In [10]:
# Change "date" to the exact date-column name shown above if needed
DATE_COLUMN = "date"

# Convert the date column only for validation
# Any invalid date values become NaT temporarily
parsed_dates = pd.to_datetime(train_df[DATE_COLUMN], errors="coerce")

# Count missing dates from the original column
missing_dates = train_df[DATE_COLUMN].isnull().sum()

# Count values that cannot be converted into dates
invalid_dates = (
    train_df[DATE_COLUMN].notnull() & parsed_dates.isnull()
).sum()

# Print results
print("Missing dates:", missing_dates)
print("Invalid date values:", invalid_dates)

# Show the earliest and latest valid date
print("Earliest valid date:", parsed_dates.min())
print("Latest valid date:", parsed_dates.max())

# Show examples if invalid dates were found
if invalid_dates > 0:
    print("\nRows with invalid dates:")
    display(train_df[parsed_dates.isnull() & train_df[DATE_COLUMN].notnull()].head())
else:
    print("\nPASS: All non-missing date values can be read as dates.")

Missing dates: 0
Invalid date values: 0
Earliest valid date: 2013-01-01 00:00:00
Latest valid date: 2017-12-31 00:00:00

PASS: All non-missing date values can be read as dates.


In [11]:
# Create a simple table containing the most important validation results
summary = pd.DataFrame({
    "Check": [
        "Train rows",
        "Test rows",
        "Train missing values",
        "Test missing values",
        "Train duplicate rows",
        "Test duplicate rows",
        "Missing sales values",
        "Non-numeric sales values",
        "Negative sales values"
    ],
    "Result": [
        len(train_df),
        len(test_df),
        train_df.isnull().sum().sum(),
        test_df.isnull().sum().sum(),
        train_duplicates,
        test_duplicates,
        missing_sales,
        non_numeric_sales,
        negative_sales
    ]
})

# Display the summary report
display(summary)

,Check,Result
0,Train rows,913000
1,Test rows,45000
2,Train missing values,0
3,Test missing values,0
4,Train duplicate rows,0
5,Test duplicate rows,0
6,Missing sales values,0
7,Non-numeric sales values,0
8,Negative sales values,0


Note: AI was used to refine code and check for bugs

In [12]:
import numpy as np

train_df["date"] = pd.to_datetime(train_df["date"])
train_df["product_category"] = "item_" + train_df["item"].astype(str)

weekly = (
    train_df.set_index("date")
    .groupby("product_category")
    .resample("W-MON")["sales"]
    .sum()
    .reset_index()
    .rename(columns={"date": "week_start", "sales": "units_sold"})
)
weekly["week_of_year"] = weekly["week_start"].dt.isocalendar().week.astype(int)
weekly["month"] = weekly["week_start"].dt.month
weekly["is_holiday_season"] = weekly["month"].isin([11, 12]).astype(int)

weekly = weekly.sort_values(["product_category", "week_start"]).reset_index(drop=True)
weekly["lag_1"] = weekly.groupby("product_category")["units_sold"].shift(1)
weekly["lag_4"] = weekly.groupby("product_category")["units_sold"].shift(4)
weekly["rolling_mean_4"] = weekly.groupby("product_category")["units_sold"].shift(1).rolling(4).mean().reset_index(level=0, drop=True)
weekly["rolling_mean_8"] = weekly.groupby("product_category")["units_sold"].shift(1).rolling(8).mean().reset_index(level=0, drop=True)
weekly = weekly.dropna().reset_index(drop=True)

cutoffs = weekly["week_start"].quantile([0.7, 0.85]).values
weekly["split"] = np.select(
    [weekly["week_start"] <= cutoffs[0], weekly["week_start"] <= cutoffs[1]],
    ["train", "val"], default="test",
)
train = weekly[weekly["split"] == "train"].drop(columns=["split"])
val = weekly[weekly["split"] == "val"].drop(columns=["split"])
test = weekly[weekly["split"] == "test"].drop(columns=["split"])

print(f"train: {len(train)} rows | val: {len(val)} rows | test: {len(test)} rows")

train: 8900 rows | val: 1900 rows | test: 1850 rows


In [13]:
!pip install -q mlflow xgboost lightgbm scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 737.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 375.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [14]:
import mlflow, mlflow.xgboost, mlflow.lightgbm
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

FEATURES = ["week_of_year", "month", "is_holiday_season", "lag_1", "lag_4", "rolling_mean_4", "rolling_mean_8"]
TARGET = "units_sold"

def build_X(d):
    dummies = pd.get_dummies(d["product_category"], prefix="product")
    return pd.concat([d[FEATURES], dummies], axis=1)

X_train, y_train = build_X(train), train[TARGET]
X_val, y_val = build_X(val), val[TARGET]
X_test, y_test = build_X(test), test[TARGET]
all_cols = sorted(set(X_train.columns) | set(X_val.columns) | set(X_test.columns))
X_train, X_val, X_test = [d.reindex(columns=all_cols, fill_value=0) for d in (X_train, X_val, X_test)]

def score(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1, None))) * 100
    return {"mae": mae, "rmse": rmse, "mape": mape}

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("ceramics_demand_forecasting")
results = []

def log_run(run_name, tag, params, m_val, m_test):
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("model_type", tag)
        mlflow.log_params(params)
        mlflow.log_metrics({f"val_{k}": v for k, v in m_val.items()})
        mlflow.log_metrics({f"test_{k}": v for k, v in m_test.items()})
        run_id = mlflow.active_run().info.run_id
    results.append({"run_name": run_name, "run_id": run_id, "val_mae": m_val["mae"], "test_mae": m_test["mae"], "test_rmse": m_test["rmse"], "test_mape": m_test["mape"]})

2026/10/04 22:42:00 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/04 22:42:00 INFO mlflow.store.db.utils: Updating database tables
2026/10/04 22:42:03 INFO mlflow.tracking.fluent: Experiment with name 'ceramics_demand_forecasting' does not exist. Creating a new experiment.


In [15]:
m_val, m_test = score(y_val, X_val["lag_4"]), score(y_test, X_test["lag_4"])
log_run("naive_lag4", "baseline", {"method": "lag_4"}, m_val, m_test)

m_val, m_test = score(y_val, X_val["rolling_mean_4"]), score(y_test, X_test["rolling_mean_4"])
log_run("moving_average", "baseline", {"method": "rolling_mean_4"}, m_val, m_test)

xgb = XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, random_state=42)
xgb.fit(X_train, y_train)
log_run("xgboost_v1", "candidate", xgb.get_params(), score(y_val, xgb.predict(X_val)), score(y_test, xgb.predict(X_test)))

lgbm = LGBMRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, random_state=42, verbose=-1)
lgbm.fit(X_train, y_train)
log_run("lightgbm_v1", "candidate", lgbm.get_params(), score(y_val, lgbm.predict(X_val)), score(y_test, lgbm.predict(X_test))
)
comparison = pd.DataFrame(results).sort_values("val_mae")
display(comparison)

,run_name,run_id,val_mae,test_mae,test_rmse,test_mape
2,xgboost_v1,a02e0b55941e4dfcacae41f45b131f8d,117.874115,123.997871,197.448955,3.037078
3,lightgbm_v1,f801f8a3a6514e818a355369afd9cbfd,122.001581,124.433427,198.774837,3.089971
1,moving_average,e390630f97c74c1fa7d8571dd35e73ed,283.284868,271.342568,377.557976,6.889252
0,naive_lag4,ad7575814b12427fb9b2b409237d9984,439.618421,435.278919,568.405467,11.034899


In [16]:
# --- 4.1 Save and version the processed data ---
import hashlib, os
from datetime import datetime, timezone

# Makes a fingerprint of a file (changes if the file changes)
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# Save the weekly table to a file
os.makedirs("processed_data", exist_ok=True)
PROCESSED_PATH = "processed_data/weekly_features.parquet"
weekly.to_parquet(PROCESSED_PATH, index=False)

# Fingerprint the raw and processed data
RAW_HASH = sha256_of("raw_data/train.csv")
PROCESSED_HASH = sha256_of(PROCESSED_PATH)

# Short version name for the data
DATASET_VERSION = f"weekly_v1_{PROCESSED_HASH[:10]}"

print("Processed rows:", len(weekly))
print("Raw data sha256:      ", RAW_HASH[:16], "...")
print("Processed data sha256:", PROCESSED_HASH[:16], "...")
print("Dataset version:      ", DATASET_VERSION)

Processed rows: 12650
Raw data sha256:       038f25690a65149c ...
Processed data sha256: 16dca28ce763573b ...
Dataset version:       weekly_v1_16dca28ce7


In [17]:
# --- 4.2 Save the trained models and register them ---
import warnings; warnings.filterwarnings("ignore")   # hide MLflow's harmless notes
import mlflow, mlflow.xgboost, mlflow.lightgbm
from mlflow import MlflowClient

MODEL_NAME = "ceramics_weekly_demand"
client = MlflowClient()

# Only the two trained models get registered (baselines are covered in Michelle's section)
candidates = {"xgboost_v1": (xgb, mlflow.xgboost), "lightgbm_v1": (lgbm, mlflow.lightgbm)}

# Michelle's results as a lookup table
res = pd.DataFrame(results).drop_duplicates("run_name", keep="last").set_index("run_name")

# Sample input saved with the model
input_example = X_val.head(5)

# Describe the training data
train_dataset = mlflow.data.from_pandas(train, source=PROCESSED_PATH, name="weekly_features_train",
                                        targets=TARGET)

# Versions already in the registry (so re-running this cell doesn't create duplicates)
try:
    existing = {mv.run_id: mv for mv in client.search_model_versions(f"name='{MODEL_NAME}'")}
except Exception:
    existing = {}

registered = {}
for run_name, (model, flavor) in candidates.items():
    run_id = res.loc[run_name, "run_id"]

    # Already registered from this run? Reuse it
    if run_id in existing:
        registered[run_name] = existing[run_id]
        print(f"{run_name} already registered as version {existing[run_id].version}")
        continue

    # Reopen Michelle's run and add the data info + the actual model
    with mlflow.start_run(run_id=run_id):
        mlflow.log_input(train_dataset, context="training")
        mlflow.set_tags({"dataset_version": DATASET_VERSION, "raw_data_sha256": RAW_HASH,
                         "processed_data_sha256": PROCESSED_HASH})
        info = flavor.log_model(model, name="model", input_example=input_example)

    # Register it (gets a version number)
    mv = mlflow.register_model(info.model_uri, MODEL_NAME)
    registered[run_name] = mv
    print(f"Registered {run_name} as {MODEL_NAME} version {mv.version}")

Successfully registered model 'ceramics_weekly_demand'.
Created version '1' of model 'ceramics_weekly_demand'.


Registered xgboost_v1 as ceramics_weekly_demand version 1
Registered lightgbm_v1 as ceramics_weekly_demand version 2


Registered model 'ceramics_weekly_demand' already exists. Creating a new version of this model...
Created version '2' of model 'ceramics_weekly_demand'.


In [18]:
# --- 4.3 Pick the champion (lowest validation error) and label everything ---

# Best baseline score (any run that isn't a real model)
baseline_runs = [r for r in res.index if r not in candidates]
baseline_best_val = res.loc[baseline_runs, "val_mae"].min()

# Rank models by validation error: lowest = champion
ranked = sorted(registered, key=lambda r: res.loc[r, "val_mae"])
champion_run, challenger_run = ranked[0], ranked[1]

# Description for the whole registered model
client.update_registered_model(
    MODEL_NAME,
    description=("Weekly units-sold forecast per product category for the handmade ceramics business. "
                 "Features: calendar (week_of_year, month, is_holiday_season), lag_1, lag_4, "
                 "rolling_mean_4, rolling_mean_8, product one-hot. Champion chosen on validation MAE."))

for run_name, mv in registered.items():
    r = res.loc[run_name]

    # Name tag: champion or challenger
    role = "champion" if run_name == champion_run else "challenger"
    client.set_registered_model_alias(MODEL_NAME, role, mv.version)

    # Description for this version
    client.update_model_version(
        MODEL_NAME, mv.version,
        description=(f"{run_name}: val MAE {r.val_mae:,.1f}, test MAE {r.test_mae:,.1f}, "
                     f"test MAPE {r.test_mape:.2f}%. Trained on {DATASET_VERSION} "
                     f"(chronological 70/15/15 split). Source run {r.run_id}."))

    # Extra labels (passed = beats the baselines)
    for k, v in {"validation_status": "passed" if r.val_mae < baseline_best_val else "failed_vs_baseline",
                 "selection_metric": "val_mae",
                 "val_mae": f"{r.val_mae:.2f}", "test_mae": f"{r.test_mae:.2f}",
                 "dataset_version": DATASET_VERSION, "model_family": run_name.split("_")[0]}.items():
        client.set_model_version_tag(MODEL_NAME, mv.version, k, v)

print(f"Champion:   {champion_run} (v{registered[champion_run].version})")
print(f"Challenger: {challenger_run} (v{registered[challenger_run].version})")
print(f"Best baseline val MAE: {baseline_best_val:,.1f}")

Champion:   xgboost_v1 (v1)
Challenger: lightgbm_v1 (v2)
Best baseline val MAE: 283.3


In [19]:
# --- 4.4 Show the registry as a table (use this in the demo) ---
rows = []
for hit in client.search_model_versions(f"name='{MODEL_NAME}'"):
    # Get full details (search skips the aliases)
    mv = client.get_model_version(MODEL_NAME, hit.version)
    rows.append({"version": int(mv.version), "aliases": ", ".join(mv.aliases) or "-",
                 "source_run": client.get_run(mv.run_id).info.run_name,
                 "val_mae": mv.tags.get("val_mae"), "test_mae": mv.tags.get("test_mae"),
                 "validation_status": mv.tags.get("validation_status"),
                 "dataset_version": mv.tags.get("dataset_version")})
registry_summary = pd.DataFrame(rows).sort_values("version")
display(registry_summary)

,version,aliases,source_run,val_mae,test_mae,validation_status,dataset_version
1,1,champion,xgboost_v1,117.87,124.00,passed,weekly_v1_16dca28ce7
0,2,challenger,lightgbm_v1,122.00,124.43,passed,weekly_v1_16dca28ce7


In [20]:
# --- 4.5 Forecast the next 4 weeks using the champion ---
HORIZON = 4  # weeks ahead

# Load the champion from the registry
champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
champ_mv = client.get_model_version_by_alias(MODEL_NAME, "champion")

# Only use full weeks (the last week in the data is incomplete)
last_date = train_df["date"].max()
history = weekly[weekly["week_start"] <= last_date]

# Each product's weekly sales history as a list
series = {p: g.sort_values("week_start")["units_sold"].astype(float).tolist()
          for p, g in history.groupby("product_category")}
last_week = history["week_start"].max()

forecast_rows = []
for step in range(1, HORIZON + 1):
    week = last_week + pd.Timedelta(weeks=step)

    # Build the same features the model trained on, for every product
    feats = []
    for p, s in series.items():
        feats.append({"product_category": p,
                      "week_of_year": int(week.isocalendar().week), "month": week.month,
                      "is_holiday_season": int(week.month in (11, 12)),
                      "lag_1": s[-1], "lag_4": s[-4],
                      "rolling_mean_4": np.mean(s[-4:]), "rolling_mean_8": np.mean(s[-8:])})
    f = pd.DataFrame(feats)

    # Match the training columns exactly
    X_f = build_X(f).reindex(columns=all_cols, fill_value=0).astype(X_train.dtypes.to_dict())

    # Predict
    preds = champion.predict(X_f)

    for p, yhat in zip(f["product_category"], preds):
        # Add the prediction to history so next week can use it
        series[p].append(float(yhat))
        forecast_rows.append({"product_category": p, "week_ending": week.date(), "horizon_weeks": step,
                              "forecast_units": round(float(yhat))})

# Put it in a table and label which model/data made it
forecast = pd.DataFrame(forecast_rows)
forecast["model_name"] = MODEL_NAME
forecast["model_version"] = champ_mv.version
forecast["dataset_version"] = DATASET_VERSION
forecast["generated_at_utc"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")

# Save to CSV
os.makedirs("forecasts", exist_ok=True)
FORECAST_PATH = f"forecasts/weekly_forecast_v{champ_mv.version}.csv"
forecast.to_csv(FORECAST_PATH, index=False)

# Log the forecast in MLflow
with mlflow.start_run(run_name="batch_forecast"):
    mlflow.set_tags({"model_type": "forecast_output", "dataset_version": DATASET_VERSION})
    mlflow.log_params({"model_uri": f"models:/{MODEL_NAME}@champion",
                       "model_version": champ_mv.version, "horizon_weeks": HORIZON,
                       "history_through": str(last_week.date())})
    mlflow.log_artifact(FORECAST_PATH)

print(f"History through week ending {last_week.date()} (last raw date {last_date.date()})")
print(f"Forecast from {MODEL_NAME} v{champ_mv.version} saved to {FORECAST_PATH}")
display(forecast.pivot(index="product_category", columns="week_ending", values="forecast_units").head(10))

History through week ending 2017-12-25 (last raw date 2017-12-31)
Forecast from ceramics_weekly_demand v1 saved to forecasts/weekly_forecast_v1.csv


week_ending,2018-01-01,2018-01-08,2018-01-15,2018-01-22
product_category,,,,
item_1,1278,1275,1281,1285
item_10,4811,4561,4461,4468
item_11,4377,4206,4162,4166
item_12,4149,4056,3966,4006
item_13,5493,5407,5446,5504
item_14,3562,3562,3571,3588
item_15,5779,5780,5806,5865
item_16,1507,1471,1450,1445
item_17,1986,1972,1981,1987
